# Fetching papers from arXiv



## Installing what we need

Two packages. `arxiv` is the client. `tqdm` just gives us a progress bar so we're not staring at a blank cell wondering if anything is happening.

In [7]:
! uv pip install arxiv tqdm pymupdf

Using Python 3.12.1 environment at: /Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/.venv
Checked 3 packages in 16ms


In [1]:
import arxiv
import fitz  # PyMuPDF
import json
import tempfile
from pathlib import Path
from tqdm import tqdm

## Where things go



In [2]:
DATA_DIR = Path("./papers")
TXT_DIR = DATA_DIR / "texts"
META_DIR = DATA_DIR / "metadata"

TXT_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

print(TXT_DIR.resolve())
print(META_DIR.resolve())

/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/arxivx/papers/texts
/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/arxivx/papers/metadata


## Writing the query

arXiv's query syntax lets you scope searches to the title, abstract, author, or category. You combine terms with `AND`, `OR`, `ANDNOT`. So `abs:"retrieval augmented generation"` means the phrase retrieval augmented generation appears in the abstract, and `cat:cs.CL` restricts to the computation-and-language category.

I'll search for recent RAG papers, which is a bit of a joke on ourselves but it's also useful material for the collection we're building.

In [7]:
QUERY = 'abs:"Transformers " AND cat:cs.CL'
MAX_RESULTS = 100

search = arxiv.Search(
    query=QUERY,
    max_results=MAX_RESULTS,
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending,
)

# arXiv asks clients to wait a few seconds between requests. The Client
# object takes care of that (and of retries) so we don't have to.
client = arxiv.Client(page_size=10, delay_seconds=3, num_retries=3)


## Looking before we download



In [8]:
results = list(client.results(search))

print(f"Got {len(results)} papers.\n")

for i, p in enumerate(results, 1):
    authors = ', '.join(a.name for a in p.authors[:3])
    if len(p.authors) > 3:
        authors += ', ...'
    print(f"{i}. {p.title}")
    print(f"   {authors}")
    print(f"   {p.published.date()}   {p.get_short_id()}")
    print()

Got 100 papers.

1. Toward a Functional Geometric Algebra for Natural Language Semantics
   James Pustejovsky
   2026-04-28   2604.25902v1

2. Barriers to Universal Reasoning With Transformers (And How to Overcome Them)
   Oliver Kraus, Yash Sarrof, Yuekun Yao, ...
   2026-04-28   2604.25800v1

3. WhisperPipe: A Resource-Efficient Streaming Architecture for Real-Time Automatic Speech Recognition
   Erfan Ramezani, Mohammad Mahdi Giahi, Mohammad Erfan Zarabadipour, ...
   2026-04-28   2604.25611v1

4. Scaling Probabilistic Transformer via Efficient Cross-Scale Hyperparameter Transfer
   Penghao Kuang, Haoyi Wu, Kewei Tu
   2026-04-28   2604.25409v1

5. Benchmarking PyCaret AutoML Against IndoBERT Fine-Tuning for Sentiment Analysis on Indonesian IKN Twitter Data
   Mutia Alfi Mayzaroh, Dwi Fitria Ningsih, Nindi Destriani, ...
   2026-04-28   2604.25392v1

6. Wiki Dumps to Training Corpora: South Slavic Case
   Mihailo Škorić
   2026-04-28   2604.25384v1

7. Learning from Medical Entity T

Each paper object has a bunch of attributes beyond what I printed. The interesting ones are `title`, `authors`, `summary` (which is the abstract, confusingly), `categories`, `published`, `pdf_url`, and `entry_id`. We'll save most of those.

## One quick helper for IDs



In [9]:
def paper_id(paper):
    return paper.get_short_id().split('v')[0]

paper_id(results[0])

'2604.25902'

## Downloading one paper by hand

Before we loop, let's do one the slow way. I want you to see what actually happens on each step.

In [10]:
paper = results[0]
pid = paper_id(paper)

txt_path = TXT_DIR / f"{pid}.txt"
meta_path = META_DIR / f"{pid}.json"

with tempfile.TemporaryDirectory() as tmp:
    paper.download_pdf(dirpath=tmp, filename=f"{pid}.pdf")
    doc = fitz.open(f"{tmp}/{pid}.pdf")
    text = "\n".join(page.get_text() for page in doc)
    doc.close()

txt_path.write_text(text, encoding="utf-8")
print(f"{txt_path}  ({len(text):,} chars)")

papers/texts/2604.25902.txt  (170,573 chars)


In [11]:
meta = {
    "arxiv_id": pid,
    "title": paper.title,
    "authors": [a.name for a in paper.authors],
    "abstract": paper.summary,
    "categories": paper.categories,
    "primary_category": paper.primary_category,
    "published": paper.published.isoformat(),
    "updated": paper.updated.isoformat(),
    "pdf_url": paper.pdf_url,
    "entry_id": paper.entry_id,
    "doi": paper.doi,
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

# Show everything except the abstract, which is long
print(json.dumps({k: v for k, v in meta.items() if k != 'abstract'}, indent=2))

{
  "arxiv_id": "2604.25902",
  "title": "Toward a Functional Geometric Algebra for Natural Language Semantics",
  "authors": [
    "James Pustejovsky"
  ],
  "categories": [
    "cs.CL",
    "cs.AI",
    "cs.LG"
  ],
  "primary_category": "cs.CL",
  "published": "2026-04-28T17:47:46+00:00",
  "updated": "2026-04-28T17:47:46+00:00",
  "pdf_url": "https://arxiv.org/pdf/2604.25902v1",
  "entry_id": "http://arxiv.org/abs/2604.25902v1",
  "doi": null
}


Open the JSON file in a text editor if you want. It's human-readable and will make sense.

## Now the loop



In [12]:
def download(paper, txt_dir, meta_dir, overwrite=False):
    pid = paper_id(paper)
    txt_path = txt_dir / f"{pid}.txt"
    meta_path = meta_dir / f"{pid}.json"

    if not overwrite and txt_path.exists() and meta_path.exists():
        return pid, "skip"

    try:
        with tempfile.TemporaryDirectory() as tmp:
            paper.download_pdf(dirpath=tmp, filename=f"{pid}.pdf")
            doc = fitz.open(f"{tmp}/{pid}.pdf")
            text = "\n".join(page.get_text() for page in doc)
            doc.close()
        txt_path.write_text(text, encoding="utf-8")

        meta = {
            "arxiv_id": pid,
            "title": paper.title,
            "authors": [a.name for a in paper.authors],
            "abstract": paper.summary,
            "categories": paper.categories,
            "primary_category": paper.primary_category,
            "published": paper.published.isoformat(),
            "updated": paper.updated.isoformat(),
            "pdf_url": paper.pdf_url,
            "entry_id": paper.entry_id,
            "doi": paper.doi,
        }
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2, ensure_ascii=False)

        return pid, "ok"
    except Exception as e:
        return pid, f"error: {e}"

In [13]:
for paper in tqdm(results):
    pid, status = download(paper, TXT_DIR, META_DIR)
    tqdm.write(f"{status:>6}  {pid}")

  0%|          | 0/100 [00:00<?, ?it/s]

  1%|          | 1/100 [00:00<01:01,  1.61it/s]

    ok  2604.25902


  2%|▏         | 2/100 [00:01<01:24,  1.16it/s]

    ok  2604.25800


...

    ok  2604.12162


 99%|█████████▉| 99/100 [03:35<00:04,  4.44s/it]

    ok  2604.12138


100%|██████████| 100/100 [03:37<00:00,  2.18s/it]

    ok  2604.12128


## Checking what actually landed

Don't trust the log output. Look at the disk.

In [14]:
txts = sorted(TXT_DIR.glob("*.txt"))
metas = sorted(META_DIR.glob("*.json"))

total_kb = sum(p.stat().st_size for p in txts) / 1024

print(f"{len(txts)} text files, {len(metas)} metadata files, {total_kb:.0f} KB total\n")

for p in txts:
    print(f"  {p.name}  ({p.stat().st_size / 1024:.0f} KB)")

100 text files, 100 metadata files, 6483 KB total

  2604.12128.txt  (43 KB)
  2604.12138.txt  (72 KB)
  2604.12162.txt  (109 KB)
  2604.12195.txt  (30 KB)
  2604.12231.txt  (123 KB)
  2604.12426.txt  (57 KB)
  2604.12559.txt  (76 KB)
  2604.12610.txt  (66 KB)
  2604.12633.txt  (28 KB)
  2604.12776.txt  (68 KB)
  2604.12919.txt  (58 KB)
  2604.13556.txt  (24 KB)
  2604.13950.txt  (67 KB)
  2604.13991.txt  (46 KB)
  2604.14054.txt  (83 KB)
  2604.14430.txt  (124 KB)
  2604.14442.txt  (108 KB)
  2604.14513.txt  (49 KB)
  2604.14631.txt  (83 KB)
  2604.14799.txt  (93 KB)
  2604.14807.txt  (52 KB)
  2604.14815.txt  (50 KB)
  2604.15010.txt  (69 KB)
  2604.15180.txt  (88 KB)
  2604.15371.txt  (30 KB)
  2604.15675.txt  (68 KB)
  2604.15794.txt  (38 KB)
  2604.16004.txt  (84 KB)
  2604.16787.txt  (35 KB)
  2604.17153.txt  (61 KB)
  2604.17323.txt  (42 KB)
  2604.17656.txt  (63 KB)
  2604.17674.txt  (33 KB)
  2604.17808.txt  (48 KB)
  2604.17823.txt  (45 KB)
  2604.17837.txt  (46 KB)
  2604.17

## A catalog file

The per-paper JSON files are nice for humans but annoying to iterate over in later notebooks. So I'm going to produce one `catalog.json` that lists every paper we have along with its file paths. This is the thing the next notebook (the one where we parse the PDFs) will open first.

In [16]:
catalog = []
for mf in sorted(META_DIR.glob("*.json")):
    m = json.loads(mf.read_text(encoding="utf-8"))
    catalog.append({
        "arxiv_id": m["arxiv_id"],
        "title": m["title"],
        "authors": m["authors"],
        "published": m["published"],
        "txt_path": str(TXT_DIR / f"{m['arxiv_id']}.txt"),
        "meta_path": str(mf),
    })

catalog_path = DATA_DIR / "catalog.json"
catalog_path.write_text(json.dumps(catalog, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"{len(catalog)} entries in {catalog_path}")

100 entries in papers/catalog.json
